# Meshes

Particle components can deposit any of their attributes onto a mesh. Every attribute deposited in one call shares the same mesh geometry, and the resulting fields are stored together in a `Meshes` object attached to the component as `component.meshes`. `Gas` and particle `Stars` support meshes (black holes do not).

A few properties of the meshing machinery are worth knowing up front:

- **The domain is set by the particles.** You choose a `resolution` (the cell width) and the mesh spans the full extent of the particles' support: their SPH kernels for smoothed deposition, or their clouds for cloud-in-cell deposition. No particle ever loses any of its value off the edge of the mesh.
- **Deposition is conservative.** Each particle's weights are normalised to sum to one, so the total of an extensive field over the mesh equals the particle total to floating-point precision.
- **Many attributes, one pass.** All requested attributes are deposited in a single loop over the particles, reusing each particle's weights for every attribute.
- **Extensive and intensive fields.** Extensive attributes (e.g. masses) are stored as per-cell totals. Intensive attributes (e.g. metallicities, ages) are stored as per-cell weighted means, weighted by mass by default.
- **Uniform or adaptive.** Meshes are uniform grids by default, and can be adaptively refined by passing a refinement variable and threshold.

We start by making some example gas and star particles.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from unyt import Msun, Myr, kpc

from synthesizer.mesh import Meshes
from synthesizer.particle import Gas, Stars
from synthesizer.particle.particles import CoordinateGenerator
from synthesizer.particle.utils import calculate_smoothing_lengths

rng = np.random.default_rng(42)

# Gas particles in a Gaussian blob with SPH smoothing lengths
ngas = 20000
gas_coords = CoordinateGenerator.generate_3D_gaussian(ngas) * kpc
gas = Gas(
    masses=rng.uniform(10**5, 10**5.5, ngas) * Msun,
    metallicities=10 ** rng.normal(-2.0, 0.3, ngas),
    coordinates=gas_coords,
    smoothing_lengths=calculate_smoothing_lengths(gas_coords),
    dust_to_metal_ratio=0.3,
    redshift=1,
)

# Star particles in a more compact blob (no smoothing lengths needed)
nstars = 5000
star_coords = CoordinateGenerator.generate_3D_gaussian(nstars) * 0.5 * kpc
stars = Stars(
    initial_masses=rng.uniform(10**5, 10**5.5, nstars) * Msun,
    ages=10 ** rng.uniform(0, 3, nstars) * Myr,
    metallicities=10 ** rng.normal(-2.0, 0.2, nstars),
    coordinates=star_coords,
    redshift=1,
)

## Uniform meshes

To make meshes we call `get_meshes` on a component, passing the resolution and the attributes to deposit. Extensive attributes are passed with `extensive` and intensive attributes with `intensive`. By default the particles are deposited using their SPH kernels (the `"sph_anarchy"` kernel unless another `Kernel` or kernel name is passed with `kernel`).

In [ ]:
gas_meshes = gas.get_meshes(
    resolution=0.2 * kpc,
    extensive=("masses", "dust_masses"),
    intensive="metallicities",
)
print(gas_meshes)
print(gas.meshes is gas_meshes)

The fields can be accessed either like a dictionary or as attributes. Each field is a contiguous `unyt_array` with one value per cell, shaped `(nx, ny, nz)` for a uniform mesh.

In [ ]:
print(gas_meshes["masses"].shape, gas_meshes.masses.units)
print(list(gas_meshes.keys()))
print(gas_meshes.field_info["metallicities"])

Because deposition is conservative, the mesh totals match the particle totals exactly.

In [ ]:
print(f"Particle mass: {gas.masses.sum():.6e}")
print(f"Mesh mass:     {gas_meshes.masses.sum():.6e}")
print(
    "Relative difference:",
    f"{gas_meshes.masses.sum() / gas.masses.sum() - 1:.2e}",
)

The mesh also carries its geometry: the domain `origin` and `extent`, the `resolution`, the number of cells along each axis (`dims`), and the cell centres, widths and volumes. Extensive fields can be converted to densities with `density`.

In [ ]:
print("origin:", gas_meshes.origin)
print("extent:", gas_meshes.extent)
print("dims:", gas_meshes.dims)
print("cell volume:", gas_meshes.cell_volumes[0, 0, 0])

density = gas_meshes.density("masses").to("Msun/kpc**3")
print(f"peak density: {density.max():.3e}")

Let's plot the projected gas mass and a slice through the mass-weighted metallicity.

In [ ]:
extent = [
    gas_meshes.origin[0].to_value(kpc),
    gas_meshes.extent[0].to_value(kpc),
    gas_meshes.origin[1].to_value(kpc),
    gas_meshes.extent[1].to_value(kpc),
]
mid = gas_meshes.dims[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im = axes[0].imshow(
    np.log10(gas_meshes.masses.sum(axis=2).value.T + 1),
    origin="lower",
    extent=extent,
)
fig.colorbar(im, ax=axes[0], label=r"$\log_{10}(M_{\rm gas} / M_\odot + 1)$")
axes[0].set_title("Projected mass")
im = axes[1].imshow(
    gas_meshes.metallicities[:, :, mid].T,
    origin="lower",
    extent=extent,
)
fig.colorbar(im, ax=axes[1], label="$Z$")
axes[1].set_title("Metallicity (central slice)")
for ax in axes:
    ax.set_xlabel("x / kpc")
    ax.set_ylabel("y / kpc")
plt.show()
plt.close(fig)

### Weighting intensive fields

Intensive fields are mass-weighted by default (`masses` for gas and `initial_masses` for stars). A different weighting attribute can be given with `weights`, either as a single attribute name used for every intensive field or as a dictionary mapping field names to weighting attributes.

In [ ]:
dust_weighted = gas.get_meshes(
    resolution=0.2 * kpc,
    intensive="metallicities",
    weights="dust_masses",
)
print(dust_weighted.field_info["metallicities"])

### Adding fields to existing meshes

More fields can be deposited onto an existing geometry with `add`.

In [ ]:
gas_meshes.add(gas, extensive="dust_masses", intensive="log10metallicities")
print(gas_meshes)

## Cloud-in-cell deposition

Particles without smoothing lengths (or when kernel smoothing is not wanted) can be deposited as points with `as_points=True`. Each particle is then treated as a uniform cube with the width of the cell containing it and shared between the cells it overlaps (cloud-in-cell), which is also exactly conservative.

In [ ]:
star_meshes = stars.get_meshes(
    resolution=0.1 * kpc,
    extensive="initial_masses",
    intensive=("ages", "metallicities"),
    as_points=True,
)
print(star_meshes)
print(star_meshes.field_info["ages"])
print(
    "Relative mass difference:",
    star_meshes.initial_masses.sum() / stars.initial_masses.sum() - 1,
)

## Adaptive refinement

A uniform mesh fine enough to resolve the densest regions wastes most of its cells on empty space. Passing `refine_attr` and `refine_threshold` builds an adaptively refined mesh instead: `resolution` then sets the width of the coarsest (root) cells, and any cell whose total of `refine_attr` exceeds the threshold is split into eight children, repeatedly, until every cell is below the threshold.

Refinement of a cell also stops when:

- its children would be narrower than the smallest smoothing length contributing to it (smaller cells would add no information for smoothed deposition),
- a single particle contributes to it,
- or it has reached `max_depth` (default 10).

The fields of a refined mesh are flat arrays with one value per leaf cell.

In [ ]:
refined = gas.get_meshes(
    resolution=1.0 * kpc,
    extensive="masses",
    intensive="metallicities",
    refine_attr="masses",
    refine_threshold=5 * 10**6 * Msun,
)
print(refined)
print("field shape:", refined.masses.shape)
print("depths:", np.bincount(refined.cell_depths))
print("Relative mass difference:", refined.masses.sum() / gas.masses.sum() - 1)

To visualise the refinement we draw every leaf cell cutting the $z=0$ plane, coloured by its density.

In [ ]:
centres = refined.cell_centres.to_value(kpc)
widths = refined.cell_widths.to_value(kpc)
density = refined.density("masses").to_value("Msun/kpc**3")
in_slice = np.abs(centres[:, 2]) <= 0.5 * widths

rects = [
    Rectangle((c[0] - 0.5 * w, c[1] - 0.5 * w), w, w)
    for c, w in zip(centres[in_slice], widths[in_slice])
]
patches = PatchCollection(rects, edgecolor="k", linewidth=0.2)
patches.set_array(np.log10(density[in_slice] + 1))

fig, ax = plt.subplots(figsize=(6, 5))
ax.add_collection(patches)
ax.set_xlim(refined.origin[0].to_value(kpc), refined.extent[0].to_value(kpc))
ax.set_ylim(refined.origin[1].to_value(kpc), refined.extent[1].to_value(kpc))
ax.set_aspect("equal")
fig.colorbar(
    patches, ax=ax, label=r"$\log_{10}(\rho / M_\odot\,{\rm kpc}^{-3} + 1)$"
)
ax.set_xlabel("x / kpc")
ax.set_ylabel("y / kpc")
plt.show()
plt.close(fig)

## Sharing a geometry between components

Each component's mesh domain is set by its own particles, so meshes of different components do not generally align. To compare components cell by cell, pass `like` to deposit onto an existing geometry, including its refinement. Here we deposit the stars onto the refined gas mesh. The particles' support must lie inside the existing domain.

In [ ]:
stars_on_gas = stars.get_meshes(
    resolution=1.0 * kpc,
    extensive="initial_masses",
    as_points=True,
    like=refined,
)
print(stars_on_gas)
print(np.array_equal(stars_on_gas.cell_centres, refined.cell_centres))

# Stellar-to-gas mass ratio per cell
occupied = refined.masses > 0
ratio = stars_on_gas.initial_masses[occupied] / refined.masses[occupied]
print(f"max stellar-to-gas mass ratio: {ratio.max():.3f}")

## Working with cells and points

Uniform fields are shaped `(nx, ny, nz)` while refined fields are flat, one value per leaf. Code that should work on either can use `flat`, which returns any field as a 1D array (a free view for uniform meshes). Its ordering matches the flattened cell geometry, so a calculation over every cell is the same for both kinds of mesh. For example, the total gas mass from the density field:

In [ ]:
for m in (gas_meshes, refined):
    rho = m.density("masses").reshape(-1)
    volumes = m.cell_volumes.reshape(-1)
    print(f"{(rho * volumes).sum().to('Msun'):.6e}", m.flat("masses").shape)

To evaluate fields at arbitrary positions, `cell_index` returns the flat index of the cell containing each point (or -1 outside the domain). No separate tree is needed: uniform meshes use index arithmetic and refined meshes descend their own refinement tree. `sample` wraps this, returning the value of the containing cell for each point (zero outside the domain, where nothing was deposited), finding the cells once for every requested field.

Here we look up the gas metallicity and gas density in the cells containing each star particle.

In [ ]:
idx = refined.cell_index(stars.coordinates)
gas_density_at_stars = refined.density("masses").reshape(-1)[idx]
gas_Z_at_stars = refined.sample(stars.coordinates, "metallicities")

fig, ax = plt.subplots()
ax.scatter(
    gas_density_at_stars.to_value("Msun/kpc**3"),
    gas_Z_at_stars,
    s=2,
)
ax.set_xscale("log")
ax.set_xlabel(r"$\rho_{\rm gas}$ at star / $M_\odot\,{\rm kpc}^{-3}$")
ax.set_ylabel(r"$Z_{\rm gas}$ at star")
plt.show()
plt.close(fig)

## Masks

A `mask` restricts meshing to a subset of particles; the domain then covers only the masked particles. For example, meshing only the young stars:

In [ ]:
young = stars.ages < 10 * Myr
young_meshes = stars.get_meshes(
    resolution=0.1 * kpc,
    extensive="initial_masses",
    as_points=True,
    mask=young,
)
print(young_meshes)
print(
    "Relative mass difference:",
    young_meshes.initial_masses.sum() / stars.initial_masses[young].sum() - 1,
)

## Threads and precision

Meshing is threaded with `nthreads` (when Synthesizer is built with OpenMP). The work is scheduled so that the order in which contributions are summed never depends on the number of threads, so the result is bit-for-bit identical for any thread count. The precision of the fields follows the global Synthesizer output dtype, or can be set with `out_dtype`. Meshes can also be built directly with `Meshes.from_particles`, which takes the same arguments as `get_meshes` but does not attach the result to the component.

In [ ]:
serial = Meshes.from_particles(gas, 0.2 * kpc, extensive="masses", nthreads=1)
threaded = Meshes.from_particles(
    gas, 0.2 * kpc, extensive="masses", nthreads=4
)
print("bit identical:", np.array_equal(serial.masses, threaded.masses))

single = Meshes.from_particles(
    gas, 0.2 * kpc, extensive="masses", out_dtype=np.float32
)
print(single.masses.dtype)

## Parametric components

Meshing parametric components is not yet supported; `Meshes.from_parametric` currently raises an `UnimplementedFunctionality` error.